# 14 - Kind Backend Mock Frontend

This notebook talks to the Kubernetes-hosted backend exactly like a lightweight frontend client.

What it does:
- verifies the Kind cluster and backend namespace
- starts a local `kubectl port-forward` if needed
- creates a thread via `POST /threads`
- streams a chat response via `POST /chat` and parses SSE frames
- reads persisted messages via `GET /threads/{id}/messages`
- optionally cleans up the thread and port-forward

Prerequisites:
- run `powershell -ExecutionPolicy Bypass -File .\k8s\kind\deploy-kind.ps1 -ClusterName dev`
- keep your Kind cluster running
- make sure `kubectl` is available on your PATH

In [1]:
import asyncio
import json
import subprocess
import time
from pathlib import Path

import httpx

BASE_URL = 'http://127.0.0.1:8000'
NAMESPACE = 'agent-framework'
SERVICE_NAME = 'agent-backend'
PF_PROCESS = None
THREAD_ID = None

In [2]:
def run_cmd(*args: str) -> str:
    result = subprocess.run(args, capture_output=True, text=True, check=True)
    return result.stdout.strip()

print('kubectl context :', run_cmd('kubectl', 'config', 'current-context'))
print('backend pods    :')
print(run_cmd('kubectl', 'get', 'pods', '-n', NAMESPACE, '-l', 'app=agent-backend', '-o', 'wide'))

kubectl context : kind-dev
backend pods    :
NAME                             READY   STATUS    RESTARTS   AGE     IP            NODE                NOMINATED NODE   READINESS GATES
agent-backend-549644cb69-lgkbg   1/1     Running   0          5m40s   10.244.0.19   dev-control-plane   <none>           <none>


In [3]:
async def wait_for_health(timeout_seconds: float = 30.0) -> bool:
    deadline = time.time() + timeout_seconds
    async with httpx.AsyncClient(timeout=3.0) as client:
        while time.time() < deadline:
            try:
                resp = await client.get(f'{BASE_URL}/health')
                if resp.status_code == 200:
                    return True
            except Exception:
                pass
            await asyncio.sleep(1.0)
    return False

async def ensure_port_forward() -> None:
    global PF_PROCESS
    if await wait_for_health(2.0):
        print('Backend already reachable at', BASE_URL)
        return

    print('Starting kubectl port-forward ...')
    PF_PROCESS = subprocess.Popen(
        ['kubectl', 'port-forward', '-n', NAMESPACE, f'svc/{SERVICE_NAME}', '8000:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if not await wait_for_health(30.0):
        output = ''
        if PF_PROCESS and PF_PROCESS.stdout:
            try:
                output = PF_PROCESS.stdout.read()
            except Exception:
                output = '<unable to read port-forward output>'
        raise RuntimeError(f'Port-forward failed to become ready. Output: {output}')

    print('Port-forward ready ->', BASE_URL)

await ensure_port_forward()

Backend already reachable at http://127.0.0.1:8000


In [4]:
async def create_thread(name: str) -> dict:
    async with httpx.AsyncClient(timeout=20.0) as client:
        resp = await client.post(f'{BASE_URL}/threads', json={'name': name})
        resp.raise_for_status()
        return resp.json()

async def list_threads() -> list[dict]:
    async with httpx.AsyncClient(timeout=20.0) as client:
        resp = await client.get(f'{BASE_URL}/threads')
        resp.raise_for_status()
        return resp.json()

thread = await create_thread('Kind Notebook Mock Frontend')
THREAD_ID = thread['id']
threads = await list_threads()
print('Created thread:', json.dumps(thread, indent=2))
print('Visible threads:', len(threads))

Created thread: {
  "id": "92e71360-f4b5-4a81-91aa-cb6f2425518b",
  "name": "Kind Notebook Mock Frontend",
  "user_id": null,
  "tags": [],
  "metadata": {},
  "created_at": "2026-03-22T17:41:22.334914Z",
  "updated_at": "2026-03-22T17:41:22.334914Z",
  "message_count": 0
}
Visible threads: 7


In [5]:
async def stream_chat(thread_id: str, user_message: str) -> tuple[list[dict], str]:
    payload = {
        'thread_id': thread_id,
        'messages': [
            {'role': 'user', 'content': user_message}
        ],
    }

    events: list[dict] = []
    text_parts: list[str] = []

    async with httpx.AsyncClient(timeout=180.0) as client:
        async with client.stream('POST', f'{BASE_URL}/chat', json=payload) as resp:
            resp.raise_for_status()
            async for line in resp.aiter_lines():
                if not line.startswith('data: '):
                    continue

                raw = line[6:]
                if raw == '[DONE]':
                    break

                event = json.loads(raw)
                events.append(event)

                if event.get('type') == 'text_delta':
                    text_parts.append(event['content'])
                    print(event['content'], end='')
                elif event.get('type') == 'error':
                    raise RuntimeError(event.get('message') or event.get('error') or 'Unknown SSE error')

    print()
    return events, ''.join(text_parts)

events, streamed_text = await stream_chat(
    THREAD_ID,
    'Reply with exactly: Kubernetes smoke test passed.'
)

print('\nEvent count:', len(events))
print('Final streamed text:', streamed_text)
assert streamed_text.strip() == 'Kubernetes smoke test passed.'

Kubernetes smoke test passed.

Event count: 7
Final streamed text: Kubernetes smoke test passed.


In [6]:
async def get_messages(thread_id: str) -> list[dict]:
    async with httpx.AsyncClient(timeout=20.0) as client:
        resp = await client.get(f'{BASE_URL}/threads/{thread_id}/messages')
        resp.raise_for_status()
        return resp.json()

messages = await get_messages(THREAD_ID)
print('Persisted messages:', len(messages))
for message in messages:
    print('-', message['type'], '|', message['name'], '|', (message.get('output') or message.get('input') or '')[:120])

Persisted messages: 2
- user_message | user | Reply with exactly: Kubernetes smoke test passed.
- assistant_message | assistant | Kubernetes smoke test passed.


In [7]:
async def delete_thread(thread_id: str) -> None:
    async with httpx.AsyncClient(timeout=20.0) as client:
        resp = await client.delete(f'{BASE_URL}/threads/{thread_id}')
        if resp.status_code not in (200, 204, 404):
            resp.raise_for_status()

if THREAD_ID:
    await delete_thread(THREAD_ID)
    print('Deleted thread:', THREAD_ID)

if PF_PROCESS and PF_PROCESS.poll() is None:
    PF_PROCESS.terminate()
    try:
        PF_PROCESS.wait(timeout=5)
    except Exception:
        PF_PROCESS.kill()
    print('Stopped notebook-managed port-forward')
else:
    print('No notebook-managed port-forward to stop')

Deleted thread: 92e71360-f4b5-4a81-91aa-cb6f2425518b
No notebook-managed port-forward to stop
